# 2Dify AI — Private Colab + Google Drive checkpoints
Run cells in order. Your source, completed chunks, and final output live in Google Drive so they survive a Colab runtime reset. Keep this notebook private.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, os

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/2Dify')
ROOT.mkdir(parents=True, exist_ok=True)
print('Drive workspace:', ROOT)

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install -U gradio_client requests

In [ ]:
REPO = Path('/content/2dify-repo')
if not REPO.exists():
    !git clone -q https://github.com/mukz3467/Multi-tool-Ai-.git /content/2dify-repo
else:
    !git -C /content/2dify-repo pull -q
WORKER = REPO / '2dify' / '2dify_worker.py'
print('Worker:', WORKER)

## Upload or select your video
First run: upload your video. It is copied into your private Google Drive folder. For a resumed run, set `INPUT` to the same Drive file path instead of uploading again.

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No video uploaded')
name = next(iter(uploaded))
INPUT = ROOT / name
shutil.move(name, INPUT)
OUTPUT = ROOT / (INPUT.stem + '_2dify_silent.mp4')
print('Input saved:', INPUT)
print('Output:', OUTPUT)
print('Checkpoint folder:', OUTPUT.parent / (OUTPUT.stem + '.2dify_chunks'))

## Convert / resume
If the runtime stops, reconnect Drive, rerun setup cells, set `INPUT` and `OUTPUT` to the same paths, then rerun this cell. Already completed chunks are skipped. Do not delete the checkpoint folder until the final output is complete.

In [ ]:
!python "$WORKER" "$INPUT" "$OUTPUT"

In [ ]:
print('Final output exists:', OUTPUT.exists())
if OUTPUT.exists():
    print('Size (MB):', round(OUTPUT.stat().st_size / 1024 / 1024, 2))
    files.download(str(OUTPUT))